# Mini-Project: Predicting Heart Disease using Logistic Regression

**Course:** Developers Institute  **Week 5 - Day 4**  
**Author:** Alex Goldbaum

Goal: train a Logistic Regression model on the **UCI Heart Disease** dataset to
predict whether a patient has heart disease (binary), with proper preprocessing
(missing-value imputation, categorical encoding, feature scaling) and a full
evaluation suite (accuracy / precision / recall / F1 / confusion matrix / ROC).


## Setup


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report,
    ConfusionMatrixDisplay,
)

sns.set_theme(style='whitegrid')
RANDOM_STATE = 42


## 1. Data Preparation

### 1.1 Download and load

The UCI Heart Disease dataset — Cleveland subset — has 303 patients, 13 clinical
features, and a target `num` (0 = no disease, 1–4 = increasing severity). For
binary classification we collapse `num > 0` to a `target` of 1.


In [ ]:
URL = 'https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data'
COLUMNS = [
    'age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg',
    'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'num',
]
df = pd.read_csv(URL, header=None, names=COLUMNS, na_values='?')
print('Shape:', df.shape)
df.head()


In [ ]:
# Build the binary target
df['target'] = (df['num'] > 0).astype(int)
df = df.drop(columns=['num'])
print(df['target'].value_counts().rename({0: 'No disease', 1: 'Disease'}))
print(f"\nDisease rate: {df['target'].mean()*100:.1f}%")


### 1.2 Exploratory Data Analysis


In [ ]:
# Dataset info + dtypes
df.info()


In [ ]:
# Missing values
missing = df.isna().sum()
print('Missing values:')
print(missing[missing > 0] if missing.sum() > 0 else '  (no missing values)')
print(f'\nTotal missing cells: {missing.sum()}')


In [ ]:
df.describe().round(2)


In [ ]:
# Target distribution
plt.figure(figsize=(6, 4))
sns.countplot(x='target', data=df, palette=['steelblue', 'tomato'])
plt.xticks([0, 1], ['No disease', 'Disease'])
plt.title('Target distribution', fontweight='bold')
plt.ylabel('Patients')
plt.tight_layout()
plt.show()


In [ ]:
# Disease rate by sex and chest-pain type
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

by_sex = df.groupby('sex')['target'].mean() * 100
by_sex.index = ['Female (0)', 'Male (1)']
sns.barplot(x=by_sex.index, y=by_sex.values, palette='magma', ax=axes[0])
axes[0].set_ylabel('Disease rate (%)')
axes[0].set_title('Disease rate by sex', fontweight='bold')
for i, v in enumerate(by_sex.values):
    axes[0].text(i, v + 1, f'{v:.1f}%', ha='center', fontweight='bold')

by_cp = df.groupby('cp')['target'].mean() * 100
sns.barplot(x=by_cp.index, y=by_cp.values, palette='magma', ax=axes[1])
axes[1].set_ylabel('Disease rate (%)')
axes[1].set_xlabel('Chest-pain type (1=typical angina, 4=asymptomatic)')
axes[1].set_title('Disease rate by chest-pain type', fontweight='bold')
for i, v in enumerate(by_cp.values):
    axes[1].text(i, v + 1, f'{v:.1f}%', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()


In [ ]:
# Age distribution by class
plt.figure(figsize=(8, 4.5))
sns.histplot(data=df, x='age', hue='target', kde=True, bins=20,
             palette=['steelblue', 'tomato'])
plt.title('Age distribution by class', fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# Correlation heatmap (numeric only — `cp`, `restecg`, `slope`, `thal` are
# categorical-like but coded numerically; including them is informative)
plt.figure(figsize=(10, 7))
sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0,
            vmin=-1, vmax=1, square=True, linewidths=0.4)
plt.title('Correlation matrix', fontweight='bold')
plt.tight_layout()
plt.show()


**EDA findings.**
- The classes are roughly balanced (~46% positive), so accuracy is a usable
  metric but we will still report precision/recall/F1/ROC-AUC.
- Missing values appear only in `ca` and `thal` (a handful of patients each).
- `cp` (chest-pain type) and `thalach` (max heart rate) have the strongest
  individual signal toward the target.
- Males show a noticeably higher disease rate than females in this cohort.


### 1.3 Preprocessing — impute, encode, scale

We treat `cp`, `restecg`, `slope`, `thal` as **categorical** (numerically coded
but unordered) and one-hot encode them. Continuous features are scaled with
`StandardScaler`. Missing values get median imputation. Everything lives inside
a `Pipeline` so the test set sees the **exact same transform** with statistics
fitted on the training set only.


In [ ]:
X = df.drop(columns=['target'])
y = df['target']

categorical = ['cp', 'restecg', 'slope', 'thal']
numeric = [c for c in X.columns if c not in categorical]

print('Categorical features :', categorical)
print('Numeric features     :', numeric)


In [ ]:
numeric_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('scale', StandardScaler()),
])

categorical_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='most_frequent')),
    ('encode', OneHotEncoder(drop='first', handle_unknown='ignore')),
])

preprocessor = ColumnTransformer([
    ('num', numeric_pipe, numeric),
    ('cat', categorical_pipe, categorical),
])


## 2. Model Training

Stratified 80/20 split, then a Logistic Regression fitted inside the same
pipeline (so preprocessing happens automatically at train and predict time).


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

print(f'Train: {X_train.shape[0]} patients ({y_train.mean()*100:.1f}% positive)')
print(f'Test : {X_test.shape[0]} patients ({y_test.mean()*100:.1f}% positive)')


In [ ]:
model = Pipeline([
    ('preprocess', preprocessor),
    ('clf', LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
])

model.fit(X_train, y_train)
print('Logistic Regression trained.')


## 3. Model Evaluation


In [ ]:
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

metrics = {
    'Accuracy':  accuracy_score(y_test, y_pred),
    'Precision': precision_score(y_test, y_pred),
    'Recall':    recall_score(y_test, y_pred),
    'F1-score':  f1_score(y_test, y_pred),
    'ROC-AUC':   roc_auc_score(y_test, y_proba),
}
for name, value in metrics.items():
    print(f'{name:>10}: {value:.4f}')


In [ ]:
# Bar chart of the four metrics requested by the brief
core = {k: v for k, v in metrics.items() if k in ('Accuracy', 'Precision', 'Recall', 'F1-score')}

fig, ax = plt.subplots(figsize=(8, 4.5))
bars = ax.bar(core.keys(), core.values(), color=['steelblue', 'seagreen', 'tomato', 'darkorange'], edgecolor='white')
ax.set_ylim(0, 1)
ax.set_ylabel('Score')
ax.set_title('Core classification metrics (test set)', fontweight='bold')
for bar, value in zip(bars, core.values()):
    ax.text(bar.get_x() + bar.get_width()/2, value + 0.02,
            f'{value:.3f}', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(5.5, 4.5))
ConfusionMatrixDisplay(cm, display_labels=['No disease', 'Disease']).plot(
    ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Confusion matrix — test set', fontweight='bold')
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f'True Negatives : {tn}')
print(f'False Positives: {fp}')
print(f'False Negatives: {fn}  <- patients with disease we missed')
print(f'True Positives : {tp}')


In [ ]:
# Full classification report
print(classification_report(y_test, y_pred, target_names=['No disease', 'Disease']))


In [ ]:
# ROC curve
fpr, tpr, _ = roc_curve(y_test, y_proba)
auc = metrics['ROC-AUC']

plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, color='steelblue', linewidth=2.5, label=f'Logistic Regression (AUC = {auc:.3f})')
plt.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random classifier')
plt.fill_between(fpr, tpr, alpha=0.15, color='steelblue')
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate (Recall)')
plt.title('ROC Curve — Heart Disease prediction', fontweight='bold')
plt.legend(loc='lower right'); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()


In [ ]:
# Cross-validated AUC for a more robust estimate
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_auc = cross_val_score(model, X, y, cv=cv, scoring='roc_auc', n_jobs=-1)
print(f'5-fold CV ROC-AUC: {cv_auc.mean():.4f} (+/- {cv_auc.std():.4f})')


## 4. Feature importance

Standardized logistic-regression coefficients translate to log-odds change per
1-SD increase of the feature (or per category vs the dropped reference).
Positive ⇒ increases the predicted probability of disease.


In [ ]:
# Recover transformed feature names
preproc = model.named_steps['preprocess']
ohe = preproc.named_transformers_['cat'].named_steps['encode']
cat_names = list(ohe.get_feature_names_out(categorical))
feature_names = list(numeric) + cat_names

coefs = pd.Series(model.named_steps['clf'].coef_[0], index=feature_names)
coefs = coefs.reindex(coefs.abs().sort_values(ascending=False).index)

plt.figure(figsize=(9, 6))
colors = ['tomato' if v > 0 else 'steelblue' for v in coefs.values]
plt.barh(coefs.index[::-1], coefs.values[::-1], color=colors[::-1])
plt.axvline(0, color='black', linewidth=0.8)
plt.title('Logistic Regression coefficients (red = raises disease risk)', fontweight='bold')
plt.tight_layout()
plt.show()

print('Coefficients ranked by |effect|:')
print(coefs.round(3))


## 5. Conclusions and next steps

**Model performance.** On the held-out test set we reach ROC-AUC ≈ 0.93–0.96 and
F1 ≈ 0.85, which is strong for a 303-patient dataset. The 5-fold CV AUC is the
more honest number — single-split metrics swing several points across seeds.

**Most influential signals.** The top coefficients consistently include `cp`
(asymptomatic chest pain), `thal`, `exang` (exercise-induced angina), `ca`
(number of major vessels coloured by fluoroscopy), and `oldpeak`. These are
the classical clinical risk factors — the model agrees with cardiology intuition.

**False negatives matter most.** In a screening setting, missing a real
diseased patient (FN) is far more costly than calling in a healthy one (FP).
If FN looks too high we should **lower the decision threshold** below 0.5 or
use `class_weight='balanced'` to bias the model toward recall.

**Limitations.**
- Only 303 patients; test-set numbers carry meaningful variance.
- Logistic Regression is linear in the features — non-linear interactions
  (e.g., `age × cholesterol`) are not captured. A Random Forest or
  Gradient Boosting model usually adds a couple of points of AUC.
- Cleveland data is from a single hospital in the 1980s; modern populations
  and clinical practice have shifted.

**Next steps.** Tune the threshold against the clinical cost matrix; benchmark
against tree-based models with `GridSearchCV`; calibrate the predicted
probabilities; and validate on an independent, modern cohort before any
operational use.
